In [1]:
import os
import zipfile

def extract_single_image_from_subfolder_in_zip(source_folder, target_folder):
    if not os.path.isdir(source_folder):
        print(f"Папка {source_folder} не существует.")
        return

    for root, dirs, files in os.walk(source_folder):
        for file in files:
            if file.endswith('.zip'):
                zip_path = os.path.join(root, file)
                archive_name = os.path.splitext(file)[0]

                try:
                    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                        all_items = zip_ref.namelist()

                        # Фильтруем только папки
                        folders = [item for item in all_items if item.endswith('/')]
                        if len(folders) != 1:
                            print(f'Пропущен архив {file}: должен содержать ровно одну папку')
                            continue

                        folder_name = folders[0]

                        # Получаем файлы внутри этой папки
                        images_inside = [
                            item for item in all_items
                            if item.startswith(folder_name) and not item.endswith('/')
                        ]

                        if len(images_inside) != 1:
                            print(f'Пропущен архив {file}: должно быть ровно одно изображение внутри папки')
                            continue

                        image_file = images_inside[0]
                        _, ext = os.path.splitext(image_file)
                        new_filename = archive_name + ext

                        # Вычисляем относительный путь для сохранения структуры
                        rel_path = os.path.relpath(root, source_folder)
                        output_dir = os.path.join(target_folder, rel_path)
                        output_path = os.path.join(output_dir, new_filename)

                        # Создаём структуру папок, но не ту, что внутри архива
                        os.makedirs(output_dir, exist_ok=True)

                        print(f'Извлечение: {file} → {output_path}')

                        # Извлекаем файл и записываем его под новым именем
                        with zip_ref.open(image_file) as src, open(output_path, 'wb') as dst:
                            dst.write(src.read())

                except Exception as e:
                    print(f'Ошибка при обработке архива {file}: {e}')

    print("✅ Все архивы обработаны.")

if __name__ == '__main__':
    source_folder = input("Введите путь к исходной папке с .zip архивами: ").strip()
    target_folder = input("Введите путь к папке для сохранения изображений: ").strip()

    extract_single_image_from_subfolder_in_zip(source_folder, target_folder)

Извлечение: 2025_02_04_14_30_40.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_30_40.png
Извлечение: 2025_02_04_14_31_10.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_31_10.png
Извлечение: 2025_02_04_14_31_46.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_31_46.png
Извлечение: 2025_02_04_14_32_24.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_32_24.png
Извлечение: 2025_02_04_14_32_57.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_32_57.png
Извлечение: 2025_02_04_14_33_46.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_33_46.png
Извлечение: 2025_02_04_14_34_21.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_34_21.png
Извлечение: 2025_02_04_14_35_03.zip → C:\projects\test\dataset_files\Изделие 1.1\Поперечный\2025_02_04_14_35_03.png
Извлечение: 2025_02_04_14_35_29.zip → C:\projects\test\dataset_files\Изд

In [1]:
import os
import zipfile

def extract_images_grouped_by_product(source_folder, target_folder):
    if not os.path.isdir(source_folder):
        print(f"Папка {source_folder} не существует.")
        return

    os.makedirs(target_folder, exist_ok=True)
    log_entries = []
    def log(msg):
        print(msg)
        log_entries.append(msg)

    for root, dirs, files in os.walk(source_folder):
        for file in files:
            if not file.endswith('.zip'):
                continue

            zip_path = os.path.join(root, file)
            archive_name = os.path.splitext(file)[0]

            # Ожидаем структуру: <source_folder>/<Изделие X.Y>/<Поперечный|Продольный>/<архив.zip>
            orientation_dir = os.path.basename(root)
            product_dir = os.path.basename(os.path.dirname(root))

            # Проверка имени изделия
            if not product_dir.startswith('Изделие '):
                log(f'[СКИП] {file}: не распознано имя изделия в пути "{product_dir}"')
                continue

            product_suffix = product_dir[len('Изделие '):].strip()
            if '.' not in product_suffix:
                log(f'[СКИП] {file}: не распознан формат номера изделия "{product_suffix}" (ожидается X.Y)')
                continue

            product_num, product_part = product_suffix.split('.', 1)
            product_num = product_num.strip()
            product_part = product_part.strip()

            # Определяем ориентацию
            orient_norm = orientation_dir.strip().lower()
            if orient_norm.startswith('попереч'):
                orient_token = 'поперечный'
            elif orient_norm.startswith('продол'):
                orient_token = 'продольный'
            else:
                log(f'[СКИП] {file}: неизвестная ориентация "{orientation_dir}"')
                continue

            try:
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    all_items = zip_ref.namelist()

                    # Фильтруем только папки (директории в zip оканчиваются на '/')
                    folders = [item for item in all_items if item.endswith('/')]
                    if len(folders) != 1:
                        log(f'[СКИП] {file}: должен содержать ровно одну папку, найдено {len(folders)}')
                        continue

                    folder_name = folders[0]

                    # Файлы внутри единственной папки
                    images_inside = [
                        item for item in all_items
                        if item.startswith(folder_name) and not item.endswith('/')
                    ]

                    if len(images_inside) != 1:
                        log(f'[СКИП] {file}: должно быть ровно одно изображение внутри папки, найдено {len(images_inside)}')
                        continue

                    image_file = images_inside[0]
                    _, ext = os.path.splitext(image_file)

                    # Имя файла по формату
                    base_filename = f'{product_num}_{product_part}_{orient_token}_{archive_name}'
                    new_filename = f'{base_filename}{ext}'

                    # Целевая папка ТОЛЬКО по номеру изделия (X)
                    output_dir = os.path.join(target_folder, f'Изделие {product_num}')
                    os.makedirs(output_dir, exist_ok=True)
                    output_path = os.path.join(output_dir, new_filename)

                    # Разрешение коллизий: создаём копию с суффиксом, логируем случай
                    if os.path.exists(output_path):
                        suffix = 1
                        while True:
                            candidate = os.path.join(output_dir, f'{base_filename}__dup{suffix}{ext}')
                            if not os.path.exists(candidate):
                                log(f'[ДОБЛИКАТ] {file}: "{new_filename}" уже существует, создано "{os.path.basename(candidate)}"')
                                output_path = candidate
                                break
                            suffix += 1

                    print(f'Извлечение: {file} → {output_path}')
                    with zip_ref.open(image_file) as src, open(output_path, 'wb') as dst:
                        dst.write(src.read())

            except Exception as e:
                log(f'[ОШИБКА] {file}: {e}')

    # Записываем лог, если есть что писать
    if log_entries:
        report_path = os.path.join(target_folder, 'extraction_report.txt')
        try:
            with open(report_path, 'w', encoding='utf-8') as rep:
                rep.write('Отчёт по проблемным случаям извлечения\n')
                rep.write(f'Источник: {source_folder}\n')
                rep.write(f'Цель: {target_folder}\n')
                rep.write('-' * 60 + '\n')
                for line in log_entries:
                    rep.write(line + '\n')
            print(f'ℹ️ Отчёт создан: {report_path}')
        except Exception as e:
            print(f'Не удалось записать отчёт: {e}')

    print("✅ Все архивы обработаны.")

if __name__ == '__main__':
    source_folder = input("Введите путь к исходной папке с .zip архивами: ").strip()
    target_folder = input("Введите путь к папке для сохранения изображений: ").strip()
    extract_images_grouped_by_product(source_folder, target_folder)

Извлечение: 2025_02_04_14_30_40.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_30_40.png
Извлечение: 2025_02_04_14_31_10.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_31_10.png
Извлечение: 2025_02_04_14_31_46.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_31_46.png
Извлечение: 2025_02_04_14_32_24.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_32_24.png
Извлечение: 2025_02_04_14_32_57.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_32_57.png
Извлечение: 2025_02_04_14_33_46.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_33_46.png
Извлечение: 2025_02_04_14_34_21.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_34_21.png
Извлечение: 2025_02_04_14_35_03.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_35_03.png
Извлечение: 2025_02_04_14_35_29.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_35_29.png
Извлечение: 2025_02_04_14_36_03.zip → dataset_files\Изделие 1\1_1_поперечный_2025_02_04_14_